# Submission: Machine Learning Pipeline dengan TensorFlow Extended (TFX)
### Prediksi Response Campaign Marketing

**Nama:** Harman Muhammad
**Dataset:** `marketing_campaign.csv`

Notebook ini membangun machine learning pipeline end-to-end menggunakan **TensorFlow Extended (TFX)** dengan `InteractiveContext`, mencakup seluruh komponen wajib: `ExampleGen`, `StatisticsGen`, `SchemaGen`, `ExampleValidator`, `Transform`, `Trainer`, `Resolver`, `Evaluator`, dan `Pusher`. Seluruh artefak pipeline disimpan dalam folder `HarmanM-pipeline`.

## 1. Dokumentasi Proyek

### 1.1 Informasi Dataset
Dataset `marketing_campaign.csv` berisi **2.240 data pelanggan** dari sebuah perusahaan retail yang pernah menerima serangkaian campaign marketing. Dataset memuat 29 kolom mencakup:
- **Data demografis**: `Year_Birth`, `Education`, `Marital_Status`, `Income`, `Kidhome`, `Teenhome`
- **Riwayat transaksi**: `Recency`, jumlah pembelanjaan per kategori produk (`MntWines`, `MntFruits`, `MntMeatProducts`, dll.)
- **Perilaku pembelian**: jumlah pembelian lewat web/catalog/store, jumlah kunjungan web
- **Histori campaign**: `AcceptedCmp1`-`AcceptedCmp5` (apakah pelanggan menerima campaign sebelumnya)
- **Target**: `Response` (1 = pelanggan menerima campaign terakhir, 0 = tidak)

### 1.2 Persoalan yang Ingin Diselesaikan
Response rate campaign marketing historis perusahaan ini sangat rendah (rata-rata di bawah 15% untuk tiap campaign). Mengirim campaign ke seluruh basis pelanggan secara massal (*mass marketing*) menghasilkan **biaya tinggi dengan konversi rendah**. Perusahaan perlu cara untuk mengidentifikasi pelanggan mana yang kemungkinan besar akan merespons sebuah campaign, sebelum campaign tersebut diluncurkan.

### 1.3 Solusi Machine Learning & Target
Solusi yang dibangun adalah **model klasifikasi biner** yang memprediksi probabilitas seorang pelanggan akan merespons (`Response` = 1) sebuah campaign marketing, berdasarkan profil demografis dan histori transaksinya. Target performa yang ingin dicapai:
- **AUC ≥ 0.80** pada data evaluasi — cukup baik untuk *ranking* pelanggan berdasarkan probabilitas respons.
- Pipeline dibangun secara modular dan dapat dijalankan ulang (*reproducible*) menggunakan TFX, sehingga mudah di-retrain saat ada data baru.

### 1.4 Metode Pengolahan Data, Arsitektur Model, dan Metrik Evaluasi
**Pengolahan data** (lihat bagian 2 - Data Preparation, dan komponen `Transform` di bagian 3.5):
- Cleaning: mengisi `Income` kosong dengan median, membuang outlier usia (>100 tahun) dan income ekstrem (>99th percentile), menstandarkan kategori kotor pada `Marital_Status`.
- Feature engineering: `Age` (dari `Year_Birth`), `Customer_Tenure_Days` (dari `Dt_Customer`), `Total_Spending`, `Total_Children`, `Total_Purchases`.
- Di dalam komponen `Transform` TFX: fitur numerik di-scale ke rentang [0,1] (`tft.scale_to_0_1`), fitur kategorikal (`Education`, `Marital_Status`) diubah menjadi index vocabulary (`tft.compute_and_apply_vocabulary`).

**Arsitektur model** (lihat `marketing_trainer.py`, komponen `Trainer` bagian 3.6):
- Model Keras *feed-forward* (DNN) sederhana:
  - Input numerik (26 fitur) digabung langsung.
  - Input kategorikal (`Education`, `Marital_Status`) melalui `Embedding` layer (dim=4) lalu di-flatten.
  - Semua fitur digabung → `Dense(64, relu)` → `Dropout(0.3)` → `Dense(32, relu)` → `Dense(1, sigmoid)`.
- Optimizer: Adam (lr=1e-3), loss: `binary_crossentropy`.
- `EarlyStopping` pada `val_auc` untuk mencegah overfitting.

**Metrik evaluasi** (komponen `Evaluator` bagian 3.8, via TensorFlow Model Analysis/TFMA):
- **AUC** (Area Under ROC Curve) — metrik utama, cocok untuk data yang imbalanced (Response=1 hanya ~15% populasi).
- **Binary Accuracy** — metrik pendamping, dengan threshold validasi minimal 0.5 dibanding baseline model sebelumnya (blessing mechanism).
- **Example Count** — memastikan jumlah data yang dievaluasi sesuai ekspektasi.

### 1.5 Performa Model yang Dihasilkan
*(Angka final dari komponen Evaluator/TFMA setelah model dilatih dengan hyperparameter hasil Tuner — dihitung dari model yang tersimpan terhadap seluruh eval set (429 contoh), lebih otoritatif dibanding log per-epoch saat training)*
- **AUC**: 0.9070
- **Binary Accuracy**: 0.9231
- **Example Count (eval set)**: 429
- Model dinyatakan **"blessed"** oleh Evaluator karena memenuhi threshold minimum accuracy (≥ 0.5) dan berhasil dipush oleh komponen `Pusher` ke direktori serving model.

## 2. Persiapan Environment & Data

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_model_analysis as tfma
import tfx
import os

from tfx.components import CsvExampleGen
from tfx.components import StatisticsGen
from tfx.components import SchemaGen
from tfx.components import ExampleValidator
from tfx.components import Transform
from tfx.components import Trainer
from tfx.components import Tuner
from tfx.components import Pusher
from tfx.components import Evaluator

from tfx.proto import trainer_pb2
from tfx.proto import example_gen_pb2
from tfx.proto import pusher_pb2

from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
print("TensorFlow version:", tf.__version__)
print("TFX version:", tfx.__version__)

import warnings
warnings.filterwarnings("ignore")

I0000 00:00:1788164952.621349   28506 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788164952.721402   28506 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/conda/envs/tfx-env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.21) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
I0000 00:00:1788164955.200749   28506 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/opt/cond

TensorFlow version: 2.21.0
TFX version: 1.21.0


### 2.1 Data Preparation
Sebelum masuk ke pipeline TFX, dataset mentah dibersihkan terlebih dahulu (cleaning + feature engineering dasar) dan disimpan sebagai CSV baru di folder `data/`, yang akan menjadi input untuk komponen `ExampleGen`.

In [2]:
df = pd.read_csv('marketing_campaign.csv', sep='\t')

# Cleaning
df = df.drop(columns=['Z_CostContact', 'Z_Revenue'])
df['Income'] = df['Income'].fillna(df['Income'].median())
df['Age'] = 2014 - df['Year_Birth']
df = df[df['Age'] <= 100]
income_cap = df['Income'].quantile(0.99)
df = df[df['Income'] <= income_cap]
df['Marital_Status'] = df['Marital_Status'].replace({'Alone':'Single','Absurd':'Single','YOLO':'Single'})
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')
ref_date = df['Dt_Customer'].max()
df['Customer_Tenure_Days'] = (ref_date - df['Dt_Customer']).dt.days

# Feature engineering
mnt_cols = ['MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds']
purchase_cols = ['NumDealsPurchases','NumWebPurchases','NumCatalogPurchases','NumStorePurchases']
df['Total_Spending'] = df[mnt_cols].sum(axis=1)
df['Total_Children'] = df['Kidhome'] + df['Teenhome']
df['Total_Purchases'] = df[purchase_cols].sum(axis=1)

df = df.drop(columns=['ID', 'Year_Birth', 'Dt_Customer'])

os.makedirs('data', exist_ok=True)
df.to_csv('data/marketing_campaign_clean.csv', index=False)
print("Shape akhir:", df.shape)
df.head()

Shape akhir: (2214, 29)


,Education,Marital_Status,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,...,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response,Age,Customer_Tenure_Days,Total_Spending,Total_Children,Total_Purchases
0,Graduation,Single,58138.0,0,0,58,635,88,546,172,...,0,0,0,0,1,57,663,1617,0,25
1,Graduation,Single,46344.0,1,1,38,11,1,6,2,...,0,0,0,0,0,60,113,27,2,6
2,Graduation,Together,71613.0,0,0,26,426,49,127,111,...,0,0,0,0,0,49,312,776,0,21
3,Graduation,Together,26646.0,1,0,26,11,4,20,10,...,0,0,0,0,0,30,139,53,1,8
4,PhD,Married,58293.0,1,0,94,173,43,118,46,...,0,0,0,0,0,33,161,422,1,19


## 3. Membangun Machine Learning Pipeline dengan TFX

Seluruh komponen dijalankan menggunakan `InteractiveContext`, dan artefak pipeline disimpan pada folder `HarmanM-pipeline` (folder kerja notebook ini).

In [3]:
context = InteractiveContext(pipeline_root='pipeline_root')

### 3.1 ExampleGen
Komponen pertama dalam pipeline TFX. Bertugas membaca data mentah (CSV) dan membaginya menjadi data **train** (80%) dan **eval** (20%), lalu mengubahnya menjadi format `TFRecord` yang efisien untuk komponen-komponen berikutnya.

In [4]:
import sys 
import pandas

print(sys.executable)
print('pandas:', pandas.__version__)

/opt/conda/envs/tfx-env/bin/python
pandas: 2.3.3


In [5]:
DATA_ROOT = 'data'

output_config = example_gen_pb2.Output(
    split_config=example_gen_pb2.SplitConfig(splits=[
        example_gen_pb2.SplitConfig.Split(name='train', hash_buckets=8),
        example_gen_pb2.SplitConfig.Split(name='eval', hash_buckets=2),
    ]))

example_gen = CsvExampleGen(input_base=DATA_ROOT, output_config=output_config)
context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 16
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

### 3.2 StatisticsGen
Menghasilkan **statistik deskriptif** dari data (mean, std, distribusi nilai, jumlah missing value, dll.) untuk setiap fitur. Statistik ini digunakan oleh `SchemaGen` dan `ExampleValidator` di tahap berikutnya, dan bisa divisualisasikan langsung di notebook.

In [6]:
statistics_gen = StatisticsGen(examples=example_gen.outputs['examples'])
context.run(statistics_gen)

ExecutionResult(
    component_id: StatisticsGen
    execution_id: 17
    outputs:
        statistics: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [7]:
context.show(statistics_gen.outputs['statistics'])

### 3.3 SchemaGen
Menyimpulkan (*infer*) **skema data** secara otomatis dari statistik yang dihasilkan `StatisticsGen` — mencakup tipe data tiap fitur, domain nilai (misalnya kategori yang valid), dan apakah suatu fitur wajib ada (`required`). Skema ini menjadi acuan validasi data pada tahap berikutnya.

In [8]:
schema_gen = SchemaGen(statistics=statistics_gen.outputs['statistics'], infer_feature_shape=True)
context.run(schema_gen)

ExecutionResult(
    component_id: SchemaGen
    execution_id: 18
    outputs:
        schema: OutputChannel(artifact_type=Schema, producer_component_id=SchemaGen, output_key=schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [9]:
context.show(schema_gen.outputs['schema'])

,Type,Presence,Valency,Domain
Feature name,,,,
'AcceptedCmp1',INT,required,,-
'AcceptedCmp2',INT,required,,-
'AcceptedCmp3',INT,required,,-
'AcceptedCmp4',INT,required,,-
'AcceptedCmp5',INT,required,,-
'Age',INT,required,,-
'Complain',INT,required,,-
'Customer_Tenure_Days',INT,required,,-
'Education',STRING,required,,'Education'


,Values
Domain,
'Education',"'2n Cycle', 'Basic', 'Graduation', 'Master', 'PhD'"
'Marital_Status',"'Divorced', 'Married', 'Single', 'Together', 'Widow'"


### 3.4 ExampleValidator
Membandingkan statistik data dengan skema yang telah dibuat untuk mendeteksi **anomali** — seperti missing value yang tidak wajar, nilai di luar domain yang diharapkan, atau pergeseran distribusi data (*data drift/skew*). Ini memastikan kualitas data terjaga sebelum masuk ke tahap training.

In [10]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema'])
context.run(example_validator)

ExecutionResult(
    component_id: ExampleValidator
    execution_id: 19
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [11]:
context.show(example_validator.outputs['anomalies'])

### 3.5 Transform
Melakukan **feature engineering** dan preprocessing dalam graph TensorFlow, sehingga transformasi yang sama persis diterapkan baik saat training maupun saat serving (menghindari *training-serving skew*). Logika transformasi didefinisikan di file terpisah `marketing_transform.py`:
- Fitur numerik di-scale ke rentang [0, 1] menggunakan `tft.scale_to_0_1`.
- Fitur kategorikal (`Education`, `Marital_Status`) diubah menjadi index vocabulary menggunakan `tft.compute_and_apply_vocabulary`.

In [12]:
!cat modules/marketing_transform.py

"""
Transform module untuk TFX pipeline - Marketing Campaign Response Prediction.
Melakukan preprocessing fitur numerik (scaling) dan kategorikal (vocab/one-hot).
"""
import tensorflow as tf
import tensorflow_transform as tft

LABEL_KEY = 'Response'

# Fitur numerik yang akan di-scale ke [0,1]
NUMERIC_FEATURES = [
    'Income', 'Kidhome', 'Teenhome', 'Recency',
    'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts',
    'MntSweetProducts', 'MntGoldProds',
    'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases',
    'NumStorePurchases', 'NumWebVisitsMonth',
    'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2',
    'Complain', 'Age', 'Customer_Tenure_Days',
    'Total_Spending', 'Total_Children', 'Total_Purchases',
]

# Fitur kategorikal yang akan diubah jadi index vocabulary
CATEGORICAL_FEATURES = ['Education', 'Marital_Status']


def transformed_name(key):
    return key + '_xf'


def fill_in_missing(x):
    """Isi nilai kosong dengan 

In [13]:
transform = Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file=os.path.abspath('modules/marketing_transform.py'))
context.run(transform)

running bdist_wheel
running build
running build_py
creating build/lib
copying marketing_trainer.py -> build/lib
copying marketing_transform.py -> build/lib
copying marketing_tuner.py -> build/lib


/opt/conda/envs/tfx-env/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        This deprecation is overdue, please update your project and remove deprecated
        calls to avoid build errors in the future.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


installing to /tmp/tmprqt8cs43
running install
running install_lib
copying build/lib/marketing_transform.py -> /tmp/tmprqt8cs43/.
copying build/lib/marketing_tuner.py -> /tmp/tmprqt8cs43/.
copying build/lib/marketing_trainer.py -> /tmp/tmprqt8cs43/.
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /tmp/tmprqt8cs43/./tfx_user_code_Transform-0.0+e6ded3c88e41acc3069b8f06f7dad9218262ff3c11a934e2f557dd0703b8f1b3-py3.10.egg-info
running install_scripts
creating /tmp/tmprqt8cs43/tfx_user_code_transform-0.0+e6d

E0000 00:00:1788164976.784207   28506 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


INFO:tensorflow:Assets written to: pipeline_root/Transform/transform_graph/20/.temp_path/tftransform_tmp/9c2407a6ea834055b2a3a5160d60ff4f/assets


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 20
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

### 3.5b Tuner (Hyperparameter Tuning Otomatis)
Sebelum training model final, dijalankan **hyperparameter tuning otomatis** menggunakan KerasTuner (`RandomSearch`) untuk mencari kombinasi terbaik dari:
- `units_1` — jumlah unit di Dense layer pertama (32/64/128)
- `units_2` — jumlah unit di Dense layer kedua (16/32/64)
- `dropout` — dropout rate (0.1–0.5)
- `learning_rate` — learning rate Adam (1e-2/1e-3/1e-4)

Objective yang dioptimalkan adalah **`val_auc`** (bukan accuracy), karena data imbalanced. Hasil terbaik (`best_hyperparameters`) kemudian diteruskan ke komponen `Trainer` di tahap berikutnya, sehingga model final dilatih dengan arsitektur yang sudah dioptimalkan, bukan nilai default yang ditebak manual.

In [14]:
!cat modules/marketing_tuner.py

"""
Tuner module untuk TFX pipeline - Marketing Campaign Response Prediction.
Melakukan hyperparameter tuning otomatis menggunakan KerasTuner (Hyperband)
terhadap arsitektur DNN yang sama dipakai oleh marketing_trainer.py.
"""
from typing import NamedTuple, Dict, Any, Text

import keras_tuner as kt
import tensorflow as tf
import tensorflow_transform as tft
from tfx.components.trainer.fn_args_utils import FnArgs
from tfx.v1.components import TunerFnResult

LABEL_KEY = 'Response'

NUMERIC_FEATURES = [
    'Income', 'Kidhome', 'Teenhome', 'Recency',
    'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts',
    'MntSweetProducts', 'MntGoldProds',
    'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases',
    'NumStorePurchases', 'NumWebVisitsMonth',
    'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2',
    'Complain', 'Age', 'Customer_Tenure_Days',
    'Total_Spending', 'Total_Children', 'Total_Purchases',
]
CATEGORICAL_FEATURES = ['Education'

In [15]:
tuner = Tuner(
    module_file=os.path.abspath('modules/marketing_tuner.py'),
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(num_steps=50),
    eval_args=trainer_pb2.EvalArgs(num_steps=20))
context.run(tuner)

Trial 10 Complete [00h 00m 05s]
val_auc: 0.9092828631401062

Best val_auc So Far: 0.9157451391220093
Total elapsed time: 00h 00m 54s
Results summary
Results in pipeline_root/.temp/21/marketing_response_tuning
Showing 10 best trials
Objective(name="val_auc", direction="max")

Trial 02 summary
Hyperparameters:
units_1: 128
units_2: 64
dropout: 0.5
learning_rate: 0.01
Score: 0.9157451391220093

Trial 05 summary
Hyperparameters:
units_1: 128
units_2: 32
dropout: 0.2
learning_rate: 0.001
Score: 0.9100660085678101

Trial 08 summary
Hyperparameters:
units_1: 64
units_2: 16
dropout: 0.30000000000000004
learning_rate: 0.01
Score: 0.9098525047302246

Trial 09 summary
Hyperparameters:
units_1: 64
units_2: 64
dropout: 0.30000000000000004
learning_rate: 0.01
Score: 0.9092828631401062

Trial 03 summary
Hyperparameters:
units_1: 128
units_2: 16
dropout: 0.5
learning_rate: 0.01
Score: 0.9082942008972168

Trial 07 summary
Hyperparameters:
units_1: 128
units_2: 64
dropout: 0.2
learning_rate: 0.001
Score

ExecutionResult(
    component_id: Tuner
    execution_id: 21
    outputs:
        best_hyperparameters: OutputChannel(artifact_type=HyperParameters, producer_component_id=Tuner, output_key=best_hyperparameters, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        tuner_results: OutputChannel(artifact_type=TunerResults, producer_component_id=Tuner, output_key=tuner_results, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [16]:
# Tampilkan hyperparameter terbaik hasil pencarian Tuner
best_hp_uri = tuner.outputs['best_hyperparameters'].get()[0].uri
import json as _json
with open(os.path.join(best_hp_uri, 'best_hyperparameters.txt')) as f:
    best_hp = _json.load(f)
print("Hyperparameter terbaik hasil Tuner:")
print(_json.dumps(best_hp['values'], indent=2))

Hyperparameter terbaik hasil Tuner:
{
  "units_1": 128,
  "units_2": 64,
  "dropout": 0.5,
  "learning_rate": 0.01
}


### 3.6 Trainer
Melatih model machine learning menggunakan data hasil `Transform`, dengan hyperparameter **terbaik hasil komponen `Tuner`** di atas (bukan nilai default). Arsitektur model dan proses training didefinisikan di file terpisah `marketing_trainer.py` — sebuah DNN Keras dengan input numerik + embedding untuk fitur kategorikal, dilatih untuk memprediksi `Response` (klasifikasi biner).

In [17]:
trainer = Trainer(
    module_file=os.path.abspath('modules/marketing_trainer.py'),
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    hyperparameters=tuner.outputs['best_hyperparameters'],
    train_args=trainer_pb2.TrainArgs(num_steps=250),
    eval_args=trainer_pb2.EvalArgs(num_steps=70))
context.run(trainer)

running bdist_wheel
running build
running build_py
creating build/lib
copying marketing_trainer.py -> build/lib
copying marketing_transform.py -> build/lib
copying marketing_tuner.py -> build/lib
installing to /tmp/tmp991jq01q
running install
running install_lib


/opt/conda/envs/tfx-env/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        This deprecation is overdue, please update your project and remove deprecated
        calls to avoid build errors in the future.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


copying build/lib/marketing_transform.py -> /tmp/tmp991jq01q/.
copying build/lib/marketing_tuner.py -> /tmp/tmp991jq01q/.
copying build/lib/marketing_trainer.py -> /tmp/tmp991jq01q/.
running install_egg_info
running egg_info
creating tfx_user_code_Trainer.egg-info
writing tfx_user_code_Trainer.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Trainer.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Trainer.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
Copying tfx_user_code_Trainer.egg-info to /tmp/tmp991jq01q/./tfx_user_code_Trainer-0.0+e6ded3c88e41acc3069b8f06f7dad9218262ff3c11a934e2f557dd0703b8f1b3-py3.10.egg-info
running install_scripts
creating /tmp/tmp991jq01q/tfx_user_code_trainer-0.0+e6ded3c88e41acc3069b8f06f7dad9218262ff3c11a934e2f557dd0703b8f1b3.dist-info/WHEEL
creating 

Processing ./pipeline_root/_wheels/tfx_user_code_trainer-0.0+e6ded3c88e41acc3069b8f06f7dad9218262ff3c11a934e2f557dd0703b8f1b3-py3-none-any.whl
Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - auc: 0.7520 - binary_accuracy: 0.8556 - loss: 0.3691 - val_auc: 0.8894 - val_binary_accuracy: 0.9210 - val_loss: 0.2403
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.8933 - binary_accuracy: 0.8931 - loss: 0.2663 - val_auc: 0.9060 - val_binary_accuracy: 0.9259 - val_loss: 0.2229
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.9141 - binary_accuracy: 0.8998 - loss: 0.2423 - val_auc: 0.9099 - val_binary_accuracy: 0.9237 - val_loss: 0.2186
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.9241 - binary_accuracy: 0.9054 - loss: 0.2300 - val_auc: 0.9048 - val_binary_accuracy: 0.9210 - val_loss: 0.2243
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.9331 - binary_accuracy: 0.9111 - loss: 0.2169 - val_auc: 0.9031 - val_binary_accuracy: 0.9109 - val_

INFO:tensorflow:struct2tensor is not available.


...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: pipeline_root/Trainer/model/22/Format-Serving/assets


INFO:tensorflow:Assets written to: pipeline_root/Trainer/model/22/Format-Serving/assets


ExecutionResult(
    component_id: Trainer
    execution_id: 22
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

### 3.7 Resolver
Komponen khusus (bukan `ExampleGen`/`Transform`/dst, melainkan *special node*) yang mencari **model terbaik sebelumnya** yang sudah lolos validasi (*blessed*) untuk dijadikan **baseline pembanding** oleh `Evaluator`. Pada run pertama, resolver ini belum menemukan model sebelumnya — itu wajar, karena belum ada model yang pernah di-bless.

In [18]:
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
).with_id('latest_blessed_model_resolver')
context.run(model_resolver)

ExecutionResult(
    component_id: latest_blessed_model_resolver
    execution_id: 23
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=latest_blessed_model_resolver, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        model_blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=latest_blessed_model_resolver, output_key=model_blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

### 3.8 Evaluator
Mengevaluasi performa model yang baru dilatih menggunakan **TensorFlow Model Analysis (TFMA)**, membandingkannya dengan baseline model dari `Resolver`, dan menentukan apakah model **"blessed"** (lolos validasi) berdasarkan threshold metrik yang ditentukan (di sini: `binary_accuracy` minimal 0.5).

In [19]:
eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key='Response')],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(metrics=[
            tfma.MetricConfig(class_name='ExampleCount'),
            tfma.MetricConfig(class_name='AUC'),
            tfma.MetricConfig(class_name='BinaryAccuracy'),
            tfma.MetricConfig(
                class_name='BinaryAccuracy',
                threshold=tfma.MetricThreshold(
                    value_threshold=tfma.GenericValueThreshold(lower_bound={'value': 0.5}),
                    change_threshold=tfma.GenericChangeThreshold(
                        direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                        absolute={'value': -1e-3})))
        ])
    ])

evaluator = Evaluator(
    examples=example_gen.outputs['examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config)
context.run(evaluator)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 24
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [20]:
eval_result = evaluator.outputs['evaluation']
tfma_result = tfma.load_eval_result(eval_result.get()[0].uri)

metrics = tfma_result.get_metrics_for_slice()
print("Metrik evaluasi (TFMA - Overall):\n")
for metric_name, value in metrics.items():
    print(f"{metric_name}: {value['doubleValue']:.4f}")

Metrik evaluasi (TFMA - Overall):

example_count: 429.0000
auc: 0.9070
binary_accuracy: 0.9231


In [21]:
# Cek status blessing model
blessing_uri = evaluator.outputs['blessing'].get()[0].uri
print("Blessing artifact path:", blessing_uri)
print("Isi folder:", os.listdir(blessing_uri))
print("\nModel diberkati (blessed):", 'BLESSED' in os.listdir(blessing_uri))

Blessing artifact path: pipeline_root/Evaluator/blessing/24
Isi folder: ['BLESSED']

Model diberkati (blessed): True


### 3.9 Pusher
Komponen terakhir dalam pipeline. Jika model dinyatakan **"blessed"** oleh `Evaluator`, `Pusher` akan mendorong (push) model tersebut ke direktori serving (`serving_model/marketing-response-model`) sehingga siap digunakan untuk deployment/inference.

In [22]:
serving_model_dir = os.path.abspath('serving_model/marketing-response-model')

pusher = Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(base_directory=serving_model_dir)))
context.run(pusher)

ExecutionResult(
    component_id: Pusher
    execution_id: 25
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [23]:
pushed_model_uri = pusher.outputs['pushed_model'].get()[0].uri
print("Model berhasil di-push ke:", pushed_model_uri)
print(os.listdir(pushed_model_uri))

Model berhasil di-push ke: pipeline_root/Pusher/pushed_model/25
['assets', 'fingerprint.pb', 'saved_model.pb', 'variables']


## 4. Kesimpulan

Pipeline machine learning menggunakan TensorFlow Extended (TFX) berhasil dibangun secara end-to-end, mencakup seluruh 9 komponen wajib (`ExampleGen` hingga `Pusher`), dijalankan melalui `InteractiveContext`, dengan seluruh artefak tersimpan di folder `HarmanM-pipeline`.

**Ringkasan hasil:**
- Model DNN sederhana berhasil dilatih untuk memprediksi `Response` campaign marketing dengan **validation AUC ± 0.9** dan **validation binary accuracy ± 0.92**.
- Model lolos validasi (`blessed`) oleh `Evaluator` dan berhasil di-push ke direktori serving oleh `Pusher`, sehingga siap untuk tahap deployment (misalnya via TensorFlow Serving).
- Pipeline ini dapat dijalankan ulang secara otomatis ketika ada data pelanggan baru, menjaga konsistensi preprocessing antara training dan serving berkat komponen `Transform`.

**Potensi pengembangan lanjutan:**
- Menambahkan pipeline orkestrasi produksi (Apache Beam/Airflow/Kubeflow) menggantikan `InteractiveContext` yang bersifat eksperimental/lokal.
- Eksperimen arsitektur model lain (misalnya Gradient Boosted Trees) dan hyperparameter tuning menggunakan `Tuner` component.
- Menambahkan komponen `BulkInferrer` untuk scoring seluruh basis pelanggan secara batch sebelum campaign berikutnya diluncurkan.